# 动态权限

**常见用法**：按角色/租户限制危险工具（删除、退款、导出）；
权限名单集中一处，新工具登记进名单即自动受控，工具代码零改动。

**钩子内的做法**：
- `request.runtime.config["configurable"]["role"]` 读随请求进来的登录角色 → 查 `PERMISSIONS` 名单
- 无权限 → **不调 execute**，回填"无权限：需要 xx 角色"的 ToolMessage（`status="error"`）让模型如实转告
- 与审批的区别：规则当场拒绝，不挂起图；名单外的工具默认放行

In [9]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from langgraph.types import Command, interrupt
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool
def search_records(keyword: str) -> str:
    """搜索记录"""
    return f"找到 1 条含 '{keyword}' 的记录 : record_id : 100012"


@tool
def delete_record(record_id: str) -> str:
    """删除一条记录"""
    return f"记录 {record_id} 已删除"


# 定义不同角色的工具权限
PERMISSIONS = {
    "admin": {"delete_record", "search_records"},
    "user": {"search_records"},
}


def check_permission(request: ToolCallRequest, execute):
    """动态权限：按登录角色拦截，拒绝也回填 ToolMessage 让模型知情"""
    tc = request.tool_call
    role = request.runtime.config["configurable"]["role"]
    allowed = tc["name"] in PERMISSIONS.get(role)
    if allowed:
        return execute(request)

    return ToolMessage(
        name=tc["name"],
        content=f"没有调用工具 {tc["name"]} 的权限，已拒绝执行。",
        tool_call_id=tc["id"],
        status="error"
    )


tools = [search_records, delete_record]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


# 定义状态
class State(MessagesState):
    pass


# 工具节点：
tool_node = ToolNode(tools, wrap_tool_call=check_permission)


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

In [7]:
# role 为 user 只有查看的权限,没有删除的权限
config = {"configurable": {"thread_id": "1", "role": "user"}}
res = graph.invoke({"messages": [HumanMessage("帮我查一下关于INFO的记录并删除它")]}, config=config)
print(res)

{
    'messages': [
        HumanMessage(
            content='帮我查一下关于INFO的记录并删除它',
            additional_kwargs={},
            response_metadata={},
            id='b15e95e0-799b-4ddc-bd2d-1b5b54c35616'
        ),
        AIMessage(
            content="I'll help you search for records about INFO and delete them. Let me start by searching.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 56,
                    'prompt_tokens': 316,
                    'total_tokens': 372,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 188
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': 'edf45d9c-1551-428c-803b-c631fea72d10',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b07f-1076-7851-b9e6-5ded2ddd8eea-0',
            tool_calls=[
                {
                    'name': 'search_records',
                    'args': {'keyword': 'INFO'},
                    'id': 'call_00_1bWhyxA7h4BEhBswCa9T6465',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 316,
                'output_tokens': 56,
                'total_tokens': 372,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="找到 1 条含 'INFO' 的记录 : record_id : 100012",
            name='search_records',
            id='b37cb11b-0824-48fd-aa89-84cc3cbfbe8c',
            tool_call_id='call_00_1bWhyxA7h4BEhBswCa9T6465'
        ),
        AIMessage(
            content='Found one record. Let me delete it.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 47,
                    'prompt_tokens': 402,
                    'total_tokens': 449,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 146
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '43d0a318-6664-4401-ae78-1d88438a983c',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b07f-130b-7db3-8c3f-4e817007a37c-0',
            tool_calls=[
                {
                    'name': 'delete_record',
                    'args': {'record_id': '100012'},
                    'id': 'call_00_8teigwlAW6hAmhTNYFk28125',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 402,
                'output_tokens': 47,
                'total_tokens': 449,
                'input_token_details': {'cache_read': 256},
               

In [8]:
# role 为 admin 拥有 查询+删除 权限
config = {"configurable": {"thread_id": "2", "role": "admin"}}
res = graph.invoke({"messages": [HumanMessage("帮我查一下关于INFO的记录并删除它")]}, config=config)
print(res)

{
    'messages': [
        HumanMessage(
            content='帮我查一下关于INFO的记录并删除它',
            additional_kwargs={},
            response_metadata={},
            id='58f861db-5f76-4b32-88e3-a2df50264a5d'
        ),
        AIMessage(
            content="I'll search for the record first.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 45,
                    'prompt_tokens': 316,
                    'total_tokens': 361,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 188
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '7eeca465-a8b6-4eb8-8e9b-1f13050536f2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b07f-313f-7e32-bac0-060e2a38ffe6-0',
            tool_calls=[
                {
                    'name': 'search_records',
                    'args': {'keyword': 'INFO'},
                    'id': 'call_00_jQLjGO2ZbI3iEMt4nCSi7205',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 316,
                'output_tokens': 45,
                'total_tokens': 361,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="找到 1 条含 'INFO' 的记录 : record_id : 100012",
            name='search_records',
            id='0d3a11da-7479-4fdb-b946-88d1df89a9d7',
            tool_call_id='call_00_jQLjGO2ZbI3iEMt4nCSi7205'
        ),
        AIMessage(
            content='Found one record matching "INFO" (record_id: 100012). Deleting it now.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 58,
                    'prompt_tokens': 391,
                    'total_tokens': 449,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 135
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '4384ef14-168c-48a1-8c19-f3f87b35fbf8',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b07f-33ec-7dc2-a6d9-588fb20f6c8d-0',
            tool_calls=[
                {
                    'name': 'delete_record',
                    'args': {'record_id': '100012'},
                    'id': 'call_00_JMMok8ul8jhohHvVNWr47205',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 391,
                'output_tokens': 58,
                'total_tokens': 449,
                'input_token_details': {'cache_read': 256},
                'output_token_deta